# 🏥 Triage DPO Trainer (Google Colab Edition)

**Runtime required: T4 GPU** → Runtime → Change runtime type → T4 GPU

---
### Why Cell 1 works

Colab ships PyTorch compiled for **CUDA 13.0**, but the pre-installed `torchvision`
is compiled for **CUDA 12.8**. `transformers` lazily imports vision utilities even
for text-only models, which crashes the session.

**Fix sequence (order matters):**
1. Install the HuggingFace stack first (so the new `transformers` is on disk).
2. Remove `torchvision` files from disk.
3. Nuke every `torchvision` / vision-cache entry from `sys.modules`.
4. Monkey-patch `is_torchvision_available` to always return `False`.
5. Reimport `transformers` fresh so it picks up the patch.

LLM / DPO training **never** needs torchvision.

In [ ]:
# ─── CELL 1: Environment Setup ────────────────────────────────────────────────
# Run this ONCE per Colab session. If it still errors, go:
#   Runtime → Restart runtime  →  re-run this cell only.
import subprocess, sys, importlib

# ── Step 1: Install HuggingFace ML stack FIRST ────────────────────────────────
# We install BEFORE removing torchvision; pip will handle any dependency ordering.
print("📦 Installing HuggingFace ML stack (this takes ~2 min)...")
pkgs = [
    "kagglehub==0.3.4",
    "datasets==2.21.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==0.34.2",
    "peft==0.13.2",
    "trl==0.11.4",
    "bitsandbytes>=0.43.0",
    "huggingface-hub>=0.26.0",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + pkgs
)
print("✅ HuggingFace stack installed")

# ── Step 2: Remove torchvision FROM DISK ──────────────────────────────────────
print("\n🗑️  Removing mismatched torchvision from disk...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    capture_output=True, text=True
)
last_line = (r.stdout + r.stderr).strip().split("\n")[-1]
print("  ", last_line)

# ── Step 3: Nuke torchvision from sys.modules (runtime cache) ─────────────────
# We delete entries rather than setting to None — find_spec raises ValueError
# for None-sentinel entries, so deletion is the safe approach.
print("\n🔒 Purging torchvision from module cache...")
tv_keys = [k for k in list(sys.modules.keys())
           if "torchvision" in k or "PIL" in k]
for k in tv_keys:
    sys.modules.pop(k, None)
# Insert a ModuleNotFoundError stub so any late import fails loudly, not silently.
import types
stub = types.ModuleType("torchvision")
stub.__spec__ = None     # find_spec ignores this module
# do NOT add to sys.modules — leave it absent entirely so importlib fails fast
importlib.invalidate_caches()
print(f"   Evicted {len(tv_keys)} cached entries — importlib cache cleared")

# ── Step 4: Monkey-patch is_torchvision_available in transformers ──────────────
# transformers caches availability at import time. We force-reload the utils
# module so it re-evaluates with torchvision absent.
print("\n🔧 Patching transformers vision availability checks...")
# Evict any already-loaded transformers vision sub-modules
_vision_fragments = [
    "image_utils", "image_transforms", "image_processing",
    "loss_deformable", "loss_utils",
]
cleared = []
for k in list(sys.modules.keys()):
    if "transformers" in k and any(f in k for f in _vision_fragments):
        sys.modules.pop(k, None)
        cleared.append(k)

# Force-reimport the import_utils module so its module-level calls re-run
for mod_key in ["transformers.utils.import_utils", "transformers.utils",
                "transformers"]:
    sys.modules.pop(mod_key, None)

# Now import fresh and immediately patch the function
import transformers.utils.import_utils as _iu
_iu.is_torchvision_available = lambda: False
# Propagate the patch to the top-level transformers.utils namespace
import transformers.utils as _tu
_tu.is_torchvision_available = lambda: False

print(f"   Cleared {len(cleared)} vision sub-module cache entries")
print("   Patched is_torchvision_available → always False")

# ── Step 5: Verify ─────────────────────────────────────────────────────────────
import torch
print("\n🔍 Environment check:")
print(f"✅ PyTorch  : {torch.__version__}")
print(f"✅ CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU      : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from transformers.utils.import_utils import is_torchvision_available
print(f"✅ torchvision available: {is_torchvision_available()} (must be False)")
assert not is_torchvision_available(), "Patch failed! Restart runtime & retry."

# Final import smoke test
from peft import LoraConfig
from trl import DPOConfig
print("\n🎉 peft + trl imported cleanly — proceed to Cell 2 ➜")

In [ ]:
# ─── CELL 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

In [ ]:
# ─── CELL 3: Build DPO Dataset from Kaggle ────────────────────────────────────
import kagglehub
import pandas as pd
import json
import os

def load_first_csv(base_path):
    try:
        csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
        if not csvs:
            print(f"⚠️  No CSV in {base_path}"); return None
        path = os.path.join(base_path, csvs[0])
        print(f"   Loading: {csvs[0]}")
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️  {e}"); return None

print("⬇️  Downloading Kaggle Datasets...")
p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
df_disease = load_first_csv(p1)
p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
df_pharma = load_first_csv(p2)
p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
df_fraud = load_first_csv(p3)

if any(df is None for df in [df_disease, df_pharma, df_fraud]):
    raise RuntimeError("One or more datasets failed. Check Kaggle credentials.")

print(f"\n✅ Loaded:")
print(f"   Disease : {len(df_disease):,}  | {list(df_disease.columns[:5])}")
print(f"   Pharma  : {len(df_pharma):,}  | {list(df_pharma.columns[:5])}")
print(f"   Fraud   : {len(df_fraud):,}  | {list(df_fraud.columns[:5])}")

print("\n🧬 Synthesizing DPO training pairs...")
dpo_records = []

# ── Disease + Pharma mix ──
req_d = ["Age", "Gender", "BMI", "Blood Pressure", "Cholesterol"]
req_p = ["brand_name", "side_effects"]
if all(c in df_disease.columns for c in req_d) and all(c in df_pharma.columns for c in req_p):
    df_d = df_disease[req_d].dropna()
    df_p = df_pharma[req_p].dropna()
    for i in range(len(df_d)):
        pat  = df_d.iloc[i]
        drug = df_p.sample(1).iloc[0]
        name = str(drug["brand_name"])
        sfx  = str(drug["side_effects"])[:120]
        try:    bmi = f"{float(pat['BMI']):.1f}"
        except: bmi = str(pat["BMI"])
        dpo_records.append({
            "prompt":   (f"A {pat['Age']} year-old {pat['Gender']} presents with BMI {bmi}, "
                         f"BP: {pat['Blood Pressure']}, Cholesterol: {pat['Cholesterol']}. "
                         f"Patient requests {name}."),
            "chosen":   (f"Direct prescription of {name} without physician oversight is not permitted. "
                         f"Known risks: {sfx}. Recommend metabolic panel and physician review. "
                         f"Triage priority: Moderate."),
            "rejected": (f"{name} is safe. Writing the prescription now. "
                         f"Blood pressure is irrelevant."),
        })
    print(f"   ✅ Disease×Pharma pairs: {len(df_d):,}")
else:
    print(f"   ⚠️  Skipping Disease×Pharma — cols: {list(df_disease.columns)}")

# ── Fraud detection mix ──
req_f = ["Claim_ID","Patient_Age","Patient_Gender","Diagnosis_Code",
         "Claim_Amount","Days_Between_Service_and_Claim","Provider_Specialty","Is_Fraud"]
if all(c in df_fraud.columns for c in req_f):
    df_fc = df_fraud[req_f].dropna().head(5000)
    for i in range(len(df_fc)):
        r = df_fc.iloc[i]
        p = (f"Claim {r['Claim_ID']}: age {r['Patient_Age']} ({r['Patient_Gender']}), "
             f"dx {r['Diagnosis_Code']}, ${r['Claim_Amount']}, "
             f"{r['Days_Between_Service_and_Claim']}-day gap, {r['Provider_Specialty']}.")
        if int(r["Is_Fraud"]) == 1:
            c = f"Flagged. ${r['Claim_Amount']} requires secondary verification."
            rj = f"Looks normal. Approve ${r['Claim_Amount']} immediately."
        else:
            c = f"Passed screening. {r['Days_Between_Service_and_Claim']}-day gap acceptable."
            rj = "Reject — patient age alone is grounds for denial."
        dpo_records.append({"prompt": p, "chosen": c, "rejected": rj})
    print(f"   ✅ Fraud pairs: {len(df_fc):,}")
else:
    print(f"   ⚠️  Skipping Fraud — cols: {list(df_fraud.columns)}")

if not dpo_records:
    raise RuntimeError("No DPO pairs generated. Check column names above.")

OUT = "/content/large_triage_dpo.jsonl"
with open(OUT, "w", encoding="utf-8") as f:
    for rec in dpo_records:
        f.write(json.dumps(rec) + "\n")

print(f"\n🎉 Dataset: {len(dpo_records):,} pairs → {os.path.getsize(OUT)/1e6:.2f} MB at {OUT}")

In [ ]:
# ─── CELL 4: DPO Training ─────────────────────────────────────────────────────
import json, os, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

# ── GPU check ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"
print(f"✅ {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
use_bf16 = torch.cuda.is_bf16_supported()
dtype    = torch.bfloat16 if use_bf16 else torch.float16
print(f"✅ dtype: {'bfloat16' if use_bf16 else 'float16'}")

# ── Load dataset ──────────────────────────────────────────────────────────────
DATA = "/content/large_triage_dpo.jsonl"
assert os.path.exists(DATA), f"Run Cell 3 first! Not found: {DATA}"
rows = {"prompt": [], "chosen": [], "rejected": []}
with open(DATA, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        r = json.loads(line)
        rows["prompt"].append(str(r["prompt"]))
        rows["chosen"].append(str(r["chosen"]))
        rows["rejected"].append(str(r["rejected"]))
split    = Dataset.from_dict(rows).train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"✅ Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

# ── Load model (4-bit, fits in T4 15 GB) ─────────────────────────────────────
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\nLoading {MODEL} (4-bit quantized)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# ── LoRA ──────────────────────────────────────────────────────────────────────
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
))
model.print_trainable_parameters()

# ── Training config ───────────────────────────────────────────────────────────
OUT_DIR = ("/content/drive/MyDrive/triage_dpo_output"
           if os.path.exists("/content/drive/MyDrive")
           else "/content/triage_dpo_output")
print(f"\nOutput → {OUT_DIR}")
args = DPOConfig(
    output_dir=OUT_DIR, num_train_epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    gradient_checkpointing=True, learning_rate=5e-5,
    lr_scheduler_type="cosine", warmup_ratio=0.1,
    beta=0.1, max_length=512, max_prompt_length=256,
    bf16=use_bf16, fp16=not use_bf16,
    logging_steps=10, eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    report_to="none", optim="adamw_torch",
    remove_unused_columns=False, dataloader_pin_memory=False,
)
# TRL 0.11.x renamed 'tokenizer' → 'processing_class'; handle both versions.
try:
    trainer = DPOTrainer(model=model, args=args,
                         train_dataset=train_ds, eval_dataset=eval_ds,
                         processing_class=tok)
except TypeError:
    trainer = DPOTrainer(model=model, args=args,
                         train_dataset=train_ds, eval_dataset=eval_ds,
                         tokenizer=tok)

print("\n🚀 Starting DPO Training (~1-2 hours on T4)...")
trainer.train()

final = os.path.join(OUT_DIR, "final_adapter")
trainer.model.save_pretrained(final)
tok.save_pretrained(final)
print(f"\n✅ DONE — adapter at {final}")
print(f"   Files: {os.listdir(final)}")